# EKF-SLAM with Monte Carlo Exploration — Interactive Analysis Notebook

**Crystal Li, Valerie Liang** · Monte Carlo Methods Final Project

---

This notebook runs the full EKF-SLAM simulation pipeline and produces quantitative evaluation plots. It is a self-contained, reproducible version of `slam_analytics.py` that can be executed from top to bottom in a clean kernel.

### What this notebook does

1. **Visualises the environment** — renders the ground-truth world geometry and shows what the robot's laser scanner sees at a single timestep, illustrating how raw range-bearing readings become a point cloud.
2. **Steps through the EKF pipeline** — shows a single predict → sense → update cycle with explicit matrix values so the math is concrete.
3. **Runs the full autonomous simulation** — the robot explores the environment using Monte Carlo uncertainty-driven navigation and EKF-SLAM for localisation and mapping.
4. **Evaluates filter health** — plots pose error, NEES consistency, innovation statistics, covariance shrinkage, and feature map accuracy.
5. **Compares two environments** — repeats the run for the lab (open room) and corridor worlds to demonstrate where the Gaussian approximation holds and where it breaks down.

### How to run

Execute cells **in order from top to bottom**. The notebook uses only the packages in `requirements.txt` plus the project's own modules (`env/`, `slam/`, `montecarlo/`, `navigation/`). Run from the project root directory.

## 0. Install Dependencies

Run this cell first in a clean environment. It installs all required packages into the current kernel's Python interpreter and verifies each import.

In [ ]:
import subprocess
import sys

packages = [
    'numpy',
    'matplotlib',
    'scipy',
    'pyyaml',
]

print('Installing required packages...\n')
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--quiet'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'  ✓ {pkg}')
    else:
        print(f'  ✗ {pkg} — FAILED')
        print(result.stderr)

print('\nVerifying imports...')
import numpy as np
import matplotlib
import scipy
import yaml
print(f'  numpy      {np.__version__}')
print(f'  matplotlib {matplotlib.__version__}')
print(f'  scipy      {scipy.__version__}')
print(f'  pyyaml     {yaml.__version__}')
print('\nAll packages ready.')

## 1. Setup and Imports

We use the `inline` matplotlib backend so all figures render directly inside the notebook. The project modules are importable as long as this notebook is run from the project root directory.

In [ ]:
%matplotlib inline

import os
import sys
import time
import warnings
from IPython.display import display

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

# Suppress RuntimeWarnings (e.g. divide-by-zero in log of near-zero covariances)
# and the non-interactive canvas UserWarning from the Agg backend
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning,
                        message='.*non-interactive.*')

# Make sure we can import from the project root
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Project imports
from config          import Config
from env.world       import World
from env.robot       import Robot
from env.sensor      import Sensor
from slam.features   import FeatureExtractor
from slam.state      import SLAMState
from slam.predict    import predict
from slam.update     import (update_single, init_corner, init_line,
                              _h_corner, _h_corner_jacobian,
                              _h_line,   _h_line_jacobian)
from slam.data_assoc import associate_observations
from montecarlo.uncertainty_map import build_uncertainty_map
from navigation.selector        import GoalSelector
from navigation.slam_world      import SLAMWorld
from navigation.controller      import Controller

print('All imports OK.')
print(f'NumPy {np.__version__}')

## 2. Helper Functions

Shared utilities used throughout the notebook.

In [ ]:
# ── Angle wrapping ────────────────────────────────────────────────────────────

def _wrap(angle):
    """Wrap a scalar angle to (-pi, pi]."""
    return float((np.asarray(angle) + np.pi) % (2 * np.pi) - np.pi)

def _wrap_arr(arr):
    """Wrap an array of angles to (-pi, pi]."""
    return (np.asarray(arr) + np.pi) % (2 * np.pi) - np.pi


# ── Wall crossing detector ────────────────────────────────────────────────────

def _check_crossing(world, p0, p1):
    """True if the straight-line move p0→p1 crosses a wall."""
    diff = p1 - p0
    dist = float(np.linalg.norm(diff))
    if dist < 1e-6:
        return False
    angle  = float(np.arctan2(diff[1], diff[0]))
    result = world.ray_intersect(p0, angle, max_range=dist)
    if result is None:
        return False
    hit_dist, _ = result
    return hit_dist < dist * 0.85


# ── Combined world (GT + SLAM) for controller proximity queries ───────────────

class _CombinedWorld:
    def __init__(self, gt, slam):
        self._gt   = gt
        self._slam = slam
    def ray_intersect(self, origin, angle, max_range=30.0):
        r1 = self._gt.ray_intersect(origin, angle, max_range)
        r2 = self._slam.ray_intersect(origin, angle, max_range)
        if r1 is None: return r2
        if r2 is None: return r1
        return r1 if r1[0] <= r2[0] else r2


# ── Plot style ────────────────────────────────────────────────────────────────

STYLE = {
    'bg':      '#0f1117', 'panel':   '#161b27',
    'text':    '#e2e8f0', 'grid':    '#1e2130',
    'accent1': '#818cf8', 'accent2': '#34d399',
    'accent3': '#f59e0b', 'accent4': '#f87171',
    'accent5': '#a78bfa', 'accent6': '#38bdf8',
}

def _style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(STYLE['panel'])
    ax.tick_params(colors=STYLE['text'], labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(STYLE['grid'])
    ax.xaxis.label.set_color(STYLE['text'])
    ax.yaxis.label.set_color(STYLE['text'])
    ax.title.set_color(STYLE['text'])
    ax.grid(True, color=STYLE['grid'], linewidth=0.5, alpha=0.7)
    if title:  ax.set_title(title, fontsize=8, pad=4)
    if xlabel: ax.set_xlabel(xlabel, fontsize=7)
    if ylabel: ax.set_ylabel(ylabel, fontsize=7)

print('Helpers defined.')

## 3. Visualising the Environment and Sensor

Before running the full simulation, it helps to see what the robot is actually working with.

### 2.1 Ground-truth world geometry

The world is defined by a set of line **segments** (walls) and **corners** (wall junctions). The robot never has direct access to these — it only receives noisy sensor observations *derived* from them. The figure below shows both preset environments used in the experiments.

In [ ]:
def draw_world(ax, world, title):
    """Draw ground-truth walls and corners on ax."""
    ax.set_facecolor(STYLE['panel'])
    ax.set_aspect('equal')
    ax.set_title(title, color=STYLE['text'], fontsize=9)
    ax.tick_params(colors=STYLE['text'], labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(STYLE['grid'])
    ax.grid(True, color=STYLE['grid'], linewidth=0.4, alpha=0.5)

    for seg in world.segments:
        ax.plot([seg.p0[0], seg.p1[0]], [seg.p0[1], seg.p1[1]],
                color=STYLE['accent6'], linewidth=2.0, solid_capstyle='round')

    for corner in world.corners:
        colour = STYLE['accent3'] if corner.kind == 'convex' else STYLE['accent4']
        ax.plot(corner.pos[0], corner.pos[1], 'o',
                color=colour, markersize=7, zorder=5)

    # Legend
    ax.plot([], [], 'o', color=STYLE['accent3'], label='Convex corner')
    ax.plot([], [], 'o', color=STYLE['accent4'], label='Concave corner')
    ax.plot([], [], '-', color=STYLE['accent6'], label='Wall segment')
    ax.legend(fontsize=6, labelcolor=STYLE['text'],
              facecolor=STYLE['panel'], edgecolor=STYLE['grid'],
              loc='upper right')


fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=STYLE['bg'])
fig.suptitle('Ground-Truth Environment Geometry', color=STYLE['text'], fontsize=11)

lab_world  = World.from_preset('lab')
corr_world = World.from_preset('corridor')

draw_world(axes[0], lab_world,  'Lab world (12×8 m) — open room with interior alcove')
draw_world(axes[1], corr_world, 'Corridor world (20×3 m) — narrow corridor with branch')

plt.tight_layout()
plt.savefig('nb_world_geometry.png', dpi=120, bbox_inches='tight',
            facecolor=STYLE['bg'])
display(fig)
plt.close(fig)
print('World geometry saved to nb_world_geometry.png')

### 2.2 What the laser scanner sees

The robot carries a simulated SICK-style 2-D laser scanner with **181 rays** covering a 180° forward arc. Each ray returns:
- **range** $r_i$ — distance to the nearest wall (metres)
- **bearing** $\beta_i$ — angle relative to the robot's heading (radians)

Both readings have independent Gaussian noise added: $\sigma_r = 0.03$ m and $\sigma_\beta = 0.01$ rad.

The figure below shows a single scan from a fixed robot position, illustrating:
- The **ray fan** (lines from robot to each hit point)
- The **noisy hit points** in world frame
- The **depth discontinuities** where the segmentation algorithm splits the point cloud into separate wall segments

In [ ]:
cfg  = Config.load('config.yaml')
rng  = np.random.default_rng(42)
world = World.from_preset('lab')

# Place the robot at a characteristic interior position
robot  = Robot(x=3.0, y=2.0, theta=np.radians(30))
sensor = Sensor.from_cfg(cfg.sensor, rng=rng)
scan   = sensor.scan(robot, world)

fig, ax = plt.subplots(figsize=(10, 7), facecolor=STYLE['bg'])
ax.set_facecolor(STYLE['panel'])
ax.set_aspect('equal')
ax.set_title('Single Laser Scan — robot at (3.0, 2.0), heading 30°',
             color=STYLE['text'], fontsize=9)
ax.tick_params(colors=STYLE['text'], labelsize=7)
for sp in ax.spines.values(): sp.set_edgecolor(STYLE['grid'])
ax.grid(True, color=STYLE['grid'], linewidth=0.4, alpha=0.5)

# Ground-truth walls
for seg in world.segments:
    ax.plot([seg.p0[0], seg.p1[0]], [seg.p0[1], seg.p1[1]],
            color='#334155', linewidth=2.5, zorder=1)

# Ray fan — draw every 5th ray to avoid clutter
rx, ry = robot.x, robot.y
for i, angle in enumerate(scan.ray_angles):
    if i % 5 != 0:
        continue
    r = scan.true_ranges[i]
    if np.isfinite(r):
        ex = rx + r * np.cos(angle)
        ey = ry + r * np.sin(angle)
        ax.plot([rx, ex], [ry, ey], color='#fbbf24', linewidth=0.4,
                alpha=0.4, zorder=2)

# Noisy hit points, coloured by ray index to show angular ordering
valid = scan.valid_mask
pts   = scan.hit_xy[valid]
idxs  = np.where(valid)[0]
sc = ax.scatter(pts[:, 0], pts[:, 1],
                c=idxs, cmap='plasma', s=12, zorder=4,
                label='Noisy hit points')
plt.colorbar(sc, ax=ax, label='Ray index (0=left, 180=right)',
             shrink=0.6).ax.tick_params(labelcolor=STYLE['text'],
                                        labelsize=7)

# Robot body
robot_circle = plt.Circle((rx, ry), cfg.robot.radius,
                           color=STYLE['accent2'], zorder=6)
ax.add_patch(robot_circle)
# Heading arrow
ax.annotate('', xy=(rx + 0.6*np.cos(robot.theta),
                    ry + 0.6*np.sin(robot.theta)),
            xytext=(rx, ry),
            arrowprops=dict(arrowstyle='->', color=STYLE['accent2'],
                            lw=2.0))
ax.text(rx + 0.15, ry + 0.15, 'Robot', color=STYLE['accent2'],
        fontsize=7)

# Mark depth discontinuities — adjacent points with large range jumps
ranges = scan.ranges
gap_threshold = 0.5   # metres
disc_indices  = []
for i in range(1, len(ranges)):
    if (np.isfinite(ranges[i]) and np.isfinite(ranges[i-1]) and
            abs(ranges[i] - ranges[i-1]) > gap_threshold):
        disc_indices.append(i)

for di in disc_indices:
    if valid[di]:
        ax.plot(scan.hit_xy[di, 0], scan.hit_xy[di, 1], 'x',
                color=STYLE['accent4'], markersize=10,
                markeredgewidth=2, zorder=7)

ax.plot([], [], 'x', color=STYLE['accent4'], markersize=8,
        label=f'Depth discontinuity ({len(disc_indices)} found)')
ax.plot([], [], '-', color='#fbbf24', alpha=0.6, label='Ray fan (every 5th)')
ax.legend(fontsize=7, labelcolor=STYLE['text'],
          facecolor=STYLE['panel'], edgecolor=STYLE['grid'])

xmn, xmx, ymn, ymx = world.bounds
ax.set_xlim(xmn - 0.5, xmx + 0.5)
ax.set_ylim(ymn - 0.5, ymx + 0.5)
ax.set_xlabel('x (m)', color=STYLE['text'], fontsize=7)
ax.set_ylabel('y (m)', color=STYLE['text'], fontsize=7)

plt.tight_layout()
plt.savefig('nb_single_scan.png', dpi=120, bbox_inches='tight',
            facecolor=STYLE['bg'])
display(fig)
plt.close(fig)
print(f'Valid rays: {scan.n_valid} / {sensor.num_rays}')
print(f'Depth discontinuities detected: {len(disc_indices)}')

## 4. A Single EKF Cycle: Predict → Sense → Update

Before running the full simulation, we step through one complete EKF cycle manually and print the key matrix values. This makes the math concrete.

### The prediction step

When the robot executes a control command $(v, \omega)$ over interval $\Delta t$, the EKF propagates the pose mean via unicycle kinematics and inflates the covariance to reflect motion noise:

$$P^{(0)} = \mathbf{F}\, P\, \mathbf{F}^\top + Q$$

where $\mathbf{F} = \text{diag}(F_v, I, I, \ldots)$ is the block-diagonal Jacobian of the motion model and $Q$ is the process noise covariance.

### The update step

When a landmark observation $z$ is matched to an existing feature $k$, the Kalman gain corrects the entire joint state:

$$K = P^{(0)} H_k^\top S_k^{-1}, \qquad \hat{x} \leftarrow \hat{x} + K\nu, \qquad P \leftarrow (I-KH_k)P^{(0)}(I-KH_k)^\top + KRK^\top$$

In [ ]:
cfg   = Config.load('config.yaml')
world = World.from_preset('lab')
rng   = np.random.default_rng(0)

robot     = Robot(cfg.robot.start_x, cfg.robot.start_y,
                  cfg.robot.start_theta, cfg.robot.radius)
sensor    = Sensor.from_cfg(cfg.sensor, rng=rng)
extractor = FeatureExtractor.from_cfg(cfg, rng=rng)

slam_state = SLAMState(init_pose=robot.pose, init_pose_cov=np.zeros((3,3)))

R_corner = np.diag([cfg.ekf.R_range**2, cfg.ekf.R_bearing**2])
R_line   = np.diag([cfg.ekf.R_range**2, cfg.ekf.R_bearing**2])

# ── Step 1: First scan — initialise the map ──────────────────────────────────
scan = sensor.scan(robot, world)
corners, lines = extractor.extract(robot.pose, world, scan)
all_obs = list(corners) + list(lines)

for obs, feat_idx in associate_observations(
        slam_state, all_obs, R_corner, R_line, cfg.ekf.gate_chi2):
    R = R_corner if obs.feature_kind == 'corner' else R_line
    if feat_idx == -1:
        if obs.feature_kind == 'corner':
            init_corner(slam_state, obs.z, R, cfg.ekf.init_cov_corner)
        else:
            init_line(slam_state, obs.z, R, cfg.ekf.init_cov_line,
                      seg_p0=obs.seg_p0, seg_p1=obs.seg_p1)

print(f'After first scan: {slam_state.n_features} features initialised')
print(f'State vector dimension: {slam_state.dim}')
print(f'Initial pose: {slam_state.pose.round(3)}')
print()

# ── Step 2: Move and predict ─────────────────────────────────────────────────
v, omega, dt = 0.3, 0.2, cfg.sim.dt
robot.step(v, omega, dt)

P_before = slam_state.P[:3,:3].copy()
predict(slam_state, v, omega, dt, cfg.ekf.Q_v, cfg.ekf.Q_w)
P_after = slam_state.P[:3,:3].copy()

print('── PREDICTION STEP ─────────────────────────────────────')
print(f'  Control:  v={v} m/s,  ω={omega} rad/s,  dt={dt} s')
print(f'  New pose estimate: {slam_state.pose.round(4)}')
print(f'  Pvv trace BEFORE predict: {np.trace(P_before):.6f}')
print(f'  Pvv trace AFTER  predict: {np.trace(P_after):.6f}')
print(f'  (trace increases because motion adds uncertainty)')
print()

# ── Step 3: Scan and update ──────────────────────────────────────────────────
scan2    = sensor.scan(robot, world)
corners2, lines2 = extractor.extract(robot.pose, world, scan2)
all_obs2 = list(corners2) + list(lines2)

assoc = associate_observations(slam_state, all_obs2, R_corner, R_line,
                                cfg.ekf.gate_chi2)

n_matched = sum(1 for _, fi in assoc if fi >= 0)
n_new     = sum(1 for _, fi in assoc if fi == -1)

print('── DATA ASSOCIATION ────────────────────────────────────')
print(f'  Observations this step: {len(all_obs2)}')
print(f'  Matched to existing:    {n_matched}')
print(f'  Initialised as new:     {n_new}')
print()

# Print one update in detail
for obs, feat_idx in assoc:
    if feat_idx >= 0:
        feat = slam_state.features[feat_idx]
        if feat.kind == 'corner':
            z_hat = _h_corner(slam_state, feat_idx)
            H     = _h_corner_jacobian(slam_state, feat_idx)
        else:
            z_hat = _h_line(slam_state, feat_idx)
            H     = _h_line_jacobian(slam_state, feat_idx)
        nu = obs.z - z_hat
        nu[1] = _wrap(nu[1])
        R  = R_corner if obs.feature_kind == 'corner' else R_line
        S  = H @ slam_state.P @ H.T + R
        d2 = float(nu @ np.linalg.inv(S) @ nu)

        print('── SINGLE UPDATE (first matched observation) ───────────')
        print(f'  Feature:          {feat}')
        print(f'  Observation z:    {obs.z.round(4)}')
        print(f'  Predicted ẑ:     {z_hat.round(4)}')
        print(f'  Innovation ν:     {nu.round(4)}')
        print(f'  Innovation cov S:\n{S.round(6)}')
        print(f'  Mahalanobis d²:   {d2:.4f}  (gate={cfg.ekf.gate_chi2})')

        P_pre = slam_state.P[:3,:3].copy()
        update_single(slam_state, feat_idx, obs.z, R)
        P_post = slam_state.P[:3,:3].copy()

        print(f'  Pvv trace BEFORE update: {np.trace(P_pre):.6f}')
        print(f'  Pvv trace AFTER  update: {np.trace(P_post):.6f}')
        print(f'  (trace shrinks because observation reduces uncertainty)')
        break

print('\nSingle EKF cycle complete.')

## 5. Simulation Runner and Recorder

The `SimRecorder` accumulates per-timestep ground-truth vs. EKF quantities throughout the run. The `run_simulation` function drives the full autonomous loop:

1. **Move** — controller issues $(v, \omega)$; robot kinematics update ground-truth pose
2. **Predict** — EKF predicts new pose and inflates covariance
3. **Sense** — laser scan + feature extraction
4. **Associate + Update** — Mahalanobis gating, then Kalman update or new-feature initialisation
5. **MC uncertainty map** — sample navigable points, score by proximity to uncertain features
6. **Goal selection** — two-phase local/global selector picks the next navigation target

In [ ]:
class SimRecorder:
    """Accumulates ground-truth vs EKF metrics during the simulation loop."""

    def __init__(self):
        self.times, self.gt_x, self.gt_y, self.gt_th = [], [], [], []
        self.ekf_x, self.ekf_y, self.ekf_th = [], [], []
        self.pose_cov_trace, self.nees       = [], []
        self.innovations, self.innov_covs, self.innov_chi2 = [], [], []
        self.n_obs_total, self.n_matched, self.n_new, self.n_features = [], [], [], []
        self.goal_positions = []
        self.wall_crossings = 0
        self.path_length    = 0.0
        self._prev_pos      = None
        # finalised arrays
        self.t = self.pose_err_xy = self.pose_err_th = None

    def record_pose(self, t, gt_pose, ekf_pose, P):
        self.times.append(t)
        self.gt_x.append(float(gt_pose[0]));  self.ekf_x.append(float(ekf_pose[0]))
        self.gt_y.append(float(gt_pose[1]));  self.ekf_y.append(float(ekf_pose[1]))
        self.gt_th.append(float(gt_pose[2])); self.ekf_th.append(float(ekf_pose[2]))
        self.pose_cov_trace.append(float(P[0,0] + P[1,1] + P[2,2]))
        Pvv = P[:3,:3].copy()
        e   = gt_pose - ekf_pose;  e[2] = _wrap(e[2])
        try:
            nees_val = float(e @ np.linalg.inv(Pvv + 1e-9*np.eye(3)) @ e)
        except np.linalg.LinAlgError:
            nees_val = np.nan
        self.nees.append(nees_val)

    def record_association(self, n_obs, n_matched, n_new, n_features):
        self.n_obs_total.append(n_obs); self.n_matched.append(n_matched)
        self.n_new.append(n_new);       self.n_features.append(n_features)

    def record_innovation(self, nu, S):
        self.innovations.append(nu.copy()); self.innov_covs.append(S.copy())
        try:    chi2 = float(nu @ np.linalg.inv(S) @ nu)
        except: chi2 = np.nan
        self.innov_chi2.append(chi2)

    def record_navigation(self, pos, world, crossed):
        if crossed: self.wall_crossings += 1
        if self._prev_pos is not None:
            self.path_length += float(np.linalg.norm(pos - self._prev_pos))
        self._prev_pos = pos.copy()

    def finalise(self):
        self.t = np.array(self.times)
        gt_x  = np.array(self.gt_x);  gt_y  = np.array(self.gt_y)
        gt_th = np.array(self.gt_th)
        ex_x  = np.array(self.ekf_x); ex_y  = np.array(self.ekf_y)
        ex_th = np.array(self.ekf_th)
        self.pose_err_xy    = np.sqrt((gt_x-ex_x)**2 + (gt_y-ex_y)**2)
        self.pose_err_th    = np.abs(_wrap_arr(gt_th - ex_th))
        self.pose_cov_trace = np.array(self.pose_cov_trace)
        self.nees           = np.array(self.nees)
        self.n_features     = np.array(self.n_features)
        self.innov_chi2     = np.array(self.innov_chi2)


class GTFeatureRegistry:
    """Stores ground-truth features and matches them to EKF features post-run."""

    def __init__(self, world):
        self.gt_corners = [c.pos.copy() for c in world.corners]
        self.gt_lines   = [s.as_polar_line() for s in world.segments]

    def match_ekf_to_gt(self, state):
        ekf_corners = [(f, state.feature_mean(f.idx))
                       for f in state.features if f.kind == 'corner']
        ekf_lines   = [(f, state.feature_mean(f.idx))
                       for f in state.features if f.kind == 'line']

        corner_errors, used_c = [], set()
        for gt_pos in self.gt_corners:
            best_err, best_idx = np.inf, -1
            for i, (_, mean) in enumerate(ekf_corners):
                if i in used_c: continue
                e = float(np.linalg.norm(mean - gt_pos))
                if e < best_err: best_err, best_idx = e, i
            if best_idx >= 0 and best_err < 3.0:
                corner_errors.append(best_err); used_c.add(best_idx)

        lre, lae, used_l = [], [], set()
        for gt_rho, gt_alpha in self.gt_lines:
            best_err, best_idx = np.inf, -1
            for i, (_, mean) in enumerate(ekf_lines):
                if i in used_l: continue
                e = abs(mean[0]-gt_rho) + abs(_wrap(mean[1]-gt_alpha))
                if e < best_err: best_err, best_idx = e, i
            if best_idx >= 0 and best_err < 2.0:
                rho_e, alpha_e = ekf_lines[best_idx][1]
                lre.append(abs(rho_e - gt_rho))
                lae.append(abs(_wrap(alpha_e - gt_alpha)))
                used_l.add(best_idx)

        return {
            'corner_errors': corner_errors,
            'line_rho_errors': lre, 'line_alpha_errors': lae,
            'n_gt_corners': len(self.gt_corners),
            'n_gt_lines':   len(self.gt_lines),
            'n_matched_corners': len(corner_errors),
            'n_matched_lines':   len(lre),
        }

print('SimRecorder and GTFeatureRegistry defined.')

In [ ]:
MAX_STEPS = 8000

def run_simulation(cfg, preset, n_samples=300, seed=0, verbose=True):
    """
    Run a full autonomous EKF-SLAM exploration and return
    (recorder, gt_registry, slam_state, cfg).
    """
    rng    = np.random.default_rng(seed)
    world  = World.from_preset(preset)
    robot  = Robot(cfg.robot.start_x, cfg.robot.start_y,
                   cfg.robot.start_theta, cfg.robot.radius)
    sensor    = Sensor.from_cfg(cfg.sensor, rng=rng)
    extractor = FeatureExtractor.from_cfg(cfg, rng=rng)

    slam_state = SLAMState(init_pose=robot.pose, init_pose_cov=np.zeros((3,3)))
    selector   = GoalSelector(local_area_size=cfg.montecarlo.local_area_size,
                              min_dist=1.0,
                              dedup_dist=cfg.navigation.duplicate_thresh)
    controller = Controller(k_v=cfg.navigation.controller_k_v,
                            k_w=cfg.navigation.controller_k_w,
                            max_v=cfg.robot.max_v,
                            max_omega=cfg.robot.max_omega,
                            goal_tolerance=cfg.navigation.goal_tolerance)

    R_corner = np.diag([cfg.ekf.R_range**2, cfg.ekf.R_bearing**2])
    R_line   = np.diag([cfg.ekf.R_range**2, cfg.ekf.R_bearing**2])

    gt_registry  = GTFeatureRegistry(world)
    recorder     = SimRecorder()
    mc_rng       = np.random.default_rng(seed + 1)
    mc_every     = max(1, round(1.0 / cfg.sim.dt))
    mc_counter   = 0;  mc_cycles = 0
    last_umap    = None;  umap_fresh = False;  returning = False
    start_pos    = robot.pos.copy()
    MIN_EXPLORE_DIST = 3.0;  MIN_MC_CYCLES = 5
    dt = cfg.sim.dt

    if verbose:
        print(f"[sim] '{preset}' | seed={seed} | max_steps={MAX_STEPS}")

    for step in range(MAX_STEPS):
        t = step * dt

        _sw = SLAMWorld.from_state(slam_state)
        _cw = _CombinedWorld(world, _sw)
        v, omega = controller.step(slam_state.pose, _cw, dt, slam_world=_sw)

        prev_pos_gt = robot.pos.copy()
        robot.step(v, omega, dt)
        crossed = _check_crossing(world, prev_pos_gt, robot.pos)
        recorder.record_navigation(robot.pos, world, crossed)

        predict(slam_state, v, omega, dt, cfg.ekf.Q_v, cfg.ekf.Q_w)

        scan = sensor.scan(robot, world)
        corners, lines = extractor.extract(robot.pose, world, scan)
        all_obs = list(corners) + list(lines)

        assoc = associate_observations(slam_state, all_obs,
                                       R_corner, R_line, cfg.ekf.gate_chi2)
        n_matched = n_new = 0
        for obs, feat_idx in assoc:
            R = R_corner if obs.feature_kind == 'corner' else R_line
            if feat_idx == -1:
                n_new += 1
                if obs.feature_kind == 'corner':
                    init_corner(slam_state, obs.z, R, cfg.ekf.init_cov_corner)
                else:
                    init_line(slam_state, obs.z, R, cfg.ekf.init_cov_line,
                              seg_p0=obs.seg_p0, seg_p1=obs.seg_p1)
            else:
                n_matched += 1
                feat = slam_state.features[feat_idx]
                if feat.kind == 'corner':
                    z_hat = _h_corner(slam_state, feat_idx)
                    H     = _h_corner_jacobian(slam_state, feat_idx)
                else:
                    z_hat = _h_line(slam_state, feat_idx)
                    H     = _h_line_jacobian(slam_state, feat_idx)
                nu = obs.z - z_hat;  nu[1] = _wrap(nu[1])
                S  = H @ slam_state.P @ H.T + R
                recorder.record_innovation(nu, S)
                update_single(slam_state, feat_idx, obs.z, R)

        recorder.record_association(len(all_obs), n_matched, n_new,
                                    slam_state.n_features)
        recorder.record_pose(t, robot.pose, slam_state.pose, slam_state.P)

        mc_counter += 1
        if slam_state.n_features > 0 and mc_counter >= mc_every:
            mc_counter = 0
            last_umap  = build_uncertainty_map(
                state=slam_state, world=world, robot_pos=robot.pos,
                n_samples=n_samples, robot_radius=cfg.robot.radius,
                virtual_cov=cfg.montecarlo.virtual_cov,
                uncertainty_lo=cfg.montecarlo.uncertainty_lo,
                uncertainty_hi=cfg.montecarlo.uncertainty_hi, rng=mc_rng)
            umap_fresh = True;  mc_cycles += 1

        if last_umap is not None:
            if returning:
                controller.set_goal(start_pos)
                if controller.goal_reached(robot.pos):
                    if verbose:
                        print(f"  Returned to start at t={t:.1f}s. Done.")
                    break
            elif controller.goal_reached(robot.pos):
                selector.notify_goal_reached(robot.pos)
                controller.set_goal(None);  umap_fresh = False

            if not returning and controller.goal is None and umap_fresh:
                umap_fresh = False
                explored = (recorder.path_length >= MIN_EXPLORE_DIST
                            and mc_cycles >= MIN_MC_CYCLES)
                result = selector.select(last_umap, robot.pos)
                if result.source == 'complete' and explored:
                    if verbose:
                        print(f"  Map complete at t={t:.1f}s. Returning.")
                    returning = True
                    controller.set_goal(start_pos, world=_sw)
                elif result.source == 'complete':
                    bounds = world.bounds
                    cx = (bounds[0]+bounds[1])/2;  cy = (bounds[2]+bounds[3])/2
                    nudge = np.array([cx, cy]) + np.random.uniform(-2, 2, 2)
                    controller.set_goal(nudge, world=_sw)
                else:
                    controller.set_goal(result.goal, world=_sw)
                    recorder.goal_positions.append(result.goal.copy())

        if verbose and step % int(10.0/dt) == 0:
            err = np.sqrt((robot.x-slam_state.pose[0])**2 +
                          (robot.y-slam_state.pose[1])**2)
            print(f"  t={t:6.1f}s  feats={slam_state.n_features:3d}  "
                  f"err={err:.3f}m  crossings={recorder.wall_crossings}")
    else:
        if verbose: print(f"  Reached MAX_STEPS={MAX_STEPS}.")

    recorder.finalise()
    return recorder, gt_registry, slam_state, cfg

print('run_simulation defined.')

## 6. Run the Lab World Simulation

The **lab world** is a 12×8 m room with an L-shaped interior partial wall creating a small alcove — enough geometric diversity (convex corners, concave corners, walls of varying orientation) for the EKF to maintain a well-conditioned Gaussian posterior.

In [ ]:
cfg = Config.load('config.yaml')
t0  = time.perf_counter()

rec_lab, gt_lab, state_lab, cfg = run_simulation(
    cfg, preset='lab', n_samples=300, seed=0, verbose=True)

print(f"\nWall-clock time: {time.perf_counter()-t0:.1f}s")
print(f"EKF features:    {state_lab.n_features}")
print(f"Path length:     {rec_lab.path_length:.2f} m")
print(f"Wall crossings:  {rec_lab.wall_crossings}")

## 7. Visualise the Final SLAM Map — Lab World

This figure overlays the EKF's learned map on the ground-truth geometry. Corner features are shown as `+` markers with 2σ covariance ellipses; line features are drawn as dashed segments. The robot's driven path is shown in indigo. A well-converged map should have ellipses that are tight and centred near the true corner/wall positions.

In [ ]:
def draw_cov_ellipse(ax, mean, cov, n_std=2.0, **kwargs):
    """Draw a 2-D covariance ellipse at `mean` with `n_std` standard deviations."""
    try:
        vals, vecs = np.linalg.eigh(cov)
    except np.linalg.LinAlgError:
        return
    vals = np.maximum(vals, 1e-10)
    w = 2.0 * n_std * np.sqrt(vals[1])
    h = 2.0 * n_std * np.sqrt(vals[0])
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    defaults = dict(fill=False, linewidth=0.9, alpha=0.5, zorder=6)
    defaults.update(kwargs)
    ax.add_patch(mpatches.Ellipse(xy=(mean[0], mean[1]),
                                   width=w, height=h, angle=angle, **defaults))


def draw_slam_map(ax, world, slam_state, robot_trail, title):
    """Draw GT walls, EKF features with covariance ellipses, and robot trail."""
    ax.set_facecolor(STYLE['panel'])
    ax.set_aspect('equal')
    ax.set_title(title, color=STYLE['text'], fontsize=9)
    ax.tick_params(colors=STYLE['text'], labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor(STYLE['grid'])
    ax.grid(True, color=STYLE['grid'], linewidth=0.4, alpha=0.5)

    # Ground-truth walls (dark, as background reference)
    for seg in world.segments:
        ax.plot([seg.p0[0], seg.p1[0]], [seg.p0[1], seg.p1[1]],
                color='#334155', linewidth=3.0, zorder=1, solid_capstyle='round')

    # Robot trail
    if len(robot_trail) > 1:
        trail = np.array(robot_trail)
        ax.plot(trail[:,0], trail[:,1], color=STYLE['accent1'],
                linewidth=1.0, alpha=0.6, zorder=2, label='Robot path')

    # EKF features
    for feat in slam_state.features:
        mean = slam_state.feature_mean(feat.idx)
        cov  = slam_state.feature_cov(feat.idx)
        if feat.kind == 'corner':
            ax.plot(mean[0], mean[1], '+', color=STYLE['accent4'],
                    markersize=9, markeredgewidth=2, zorder=7)
            draw_cov_ellipse(ax, mean, cov, edgecolor=STYLE['accent4'])
        else:
            col = STYLE['accent2']
            if feat.seg_p0 is not None and feat.seg_p1 is not None:
                ax.plot([feat.seg_p0[0], feat.seg_p1[0]],
                        [feat.seg_p0[1], feat.seg_p1[1]],
                        '--', color=col, linewidth=1.5, alpha=0.7, zorder=5)
            rho, alpha = mean
            foot = np.array([rho*np.cos(alpha), rho*np.sin(alpha)])
            draw_cov_ellipse(ax, foot, cov, edgecolor=col)

    ax.plot([], [], '+', color=STYLE['accent4'], markersize=8,
            label=f'Corner features ({sum(1 for f in slam_state.features if f.kind=="corner")})')
    ax.plot([], [], '--', color=STYLE['accent2'],
            label=f'Line features ({sum(1 for f in slam_state.features if f.kind=="line")})')
    ax.legend(fontsize=6, labelcolor=STYLE['text'],
              facecolor=STYLE['panel'], edgecolor=STYLE['grid'])


lab_world = World.from_preset('lab')
# Reconstruct robot trail from recorder arrays
trail_lab = list(zip(rec_lab.gt_x, rec_lab.gt_y))

fig, ax = plt.subplots(figsize=(11, 7), facecolor=STYLE['bg'])
fig.suptitle('EKF-SLAM Final Map — Lab World', color=STYLE['text'], fontsize=11)
draw_slam_map(ax, lab_world, state_lab, trail_lab,
              f'Features: {state_lab.n_features}  |  '
              f'Path: {rec_lab.path_length:.1f} m  |  '
              f'Wall crossings: {rec_lab.wall_crossings}')
xmn, xmx, ymn, ymx = lab_world.bounds
ax.set_xlim(xmn-0.5, xmx+0.5); ax.set_ylim(ymn-0.5, ymx+0.5)
ax.set_xlabel('x (m)', color=STYLE['text'], fontsize=7)
ax.set_ylabel('y (m)', color=STYLE['text'], fontsize=7)
plt.tight_layout()
plt.savefig('nb_slam_map_lab.png', dpi=120, bbox_inches='tight',
            facecolor=STYLE['bg'])
display(fig)
plt.close(fig)
print('Map figure saved to nb_slam_map_lab.png')

## 8. Analytics Plots — Lab World

### Metrics explained

| Metric | Formula | Healthy value |
|--------|---------|---------------|
| **XY RMSE** | $\sqrt{\frac{1}{T}\sum\|\text{gt}_{xy} - \hat{x}_{xy}\|^2}$ | As small as possible |
| **NEES** | $e^\top P_{vv}^{-1} e$ where $e = x_{gt} - \hat{x}$ | $\approx 3$ (one per DOF) |
| **Innovation $\chi^2$** | $\nu^\top S^{-1} \nu$ per update | $\approx 2$ (one per observation DOF) |
| **Covariance trace** | $\text{tr}(P_{vv})$ | Decreasing over time |

**NEES** is the definitive consistency test. A value near 3 means the filter's stated uncertainty matches the true errors. NEES persistently above the 95% χ²(3) upper bound (9.3) indicates the filter is **overconfident**.

In [ ]:
def compute_summary(rec, gt_reg, slam_state, cfg):
    """Compute all summary statistics from a completed simulation run."""
    match  = gt_reg.match_ekf_to_gt(slam_state)
    t      = rec.t

    rmse_xy  = float(np.sqrt(np.mean(rec.pose_err_xy**2)))
    rmse_th  = float(np.sqrt(np.mean(rec.pose_err_th**2)))
    max_xy   = float(rec.pose_err_xy.max())
    final_xy = float(rec.pose_err_xy[-1])

    nees_valid = rec.nees[np.isfinite(rec.nees)]
    nees_mean  = float(np.mean(nees_valid)) if len(nees_valid) else np.nan
    nees_lo, nees_hi = stats.chi2.ppf([0.025, 0.975], df=3)
    frac_consistent  = float(np.mean((nees_valid >= nees_lo) &
                                      (nees_valid <= nees_hi)))

    innov_valid = rec.innov_chi2[np.isfinite(rec.innov_chi2)]
    innov_mean  = float(np.mean(innov_valid)) if len(innov_valid) else np.nan
    innov_lo, innov_hi = stats.chi2.ppf([0.025, 0.975], df=2)
    frac_innov_ok = float(np.mean((innov_valid >= innov_lo) &
                                   (innov_valid <= innov_hi)))

    ce  = match['corner_errors']
    lre = match['line_rho_errors']
    lae = match['line_alpha_errors']

    total_obs     = int(np.array(rec.n_obs_total).sum())
    total_matched = int(np.array(rec.n_matched).sum())
    match_rate    = total_matched / max(total_obs, 1)

    return {
        'rmse_xy': rmse_xy, 'rmse_th_deg': np.degrees(rmse_th),
        'max_xy': max_xy,   'final_xy': final_xy,
        'nees_mean': nees_mean, 'nees_lo': nees_lo, 'nees_hi': nees_hi,
        'frac_consistent': frac_consistent,
        'innov_mean': innov_mean, 'innov_lo': innov_lo, 'innov_hi': innov_hi,
        'frac_innov_ok': frac_innov_ok,
        'corner_rmse_cm': (float(np.sqrt(np.mean(np.array(ce)**2)))*100 if ce else np.nan),
        'line_rho_rmse_cm': (float(np.sqrt(np.mean(np.array(lre)**2)))*100 if lre else np.nan),
        'line_alpha_rmse_deg': (float(np.degrees(np.sqrt(np.mean(np.array(lae)**2)))) if lae else np.nan),
        'recall_corners': match['n_matched_corners'] / max(match['n_gt_corners'], 1),
        'recall_lines':   match['n_matched_lines']   / max(match['n_gt_lines'],   1),
        'n_ekf_features': slam_state.n_features,
        'match_rate': match_rate,
        'path_length': rec.path_length,
        'wall_crossings': rec.wall_crossings,
        'total_time': float(t[-1]) if len(t) else 0.0,
        'nees_valid': nees_valid, 'innov_valid': innov_valid,
    }


def plot_analytics(rec, summary, preset, figsize=(16, 12)):
    """Produce the 8-panel analytics figure for one simulation run."""
    t  = rec.t
    s  = summary

    fig = plt.figure(figsize=figsize, facecolor=STYLE['bg'])
    fig.suptitle(
        f"EKF-SLAM Analytics — '{preset}'  |  "
        f"duration: {s['total_time']:.0f}s  |  "
        f"path: {s['path_length']:.1f}m  |  "
        f"features: {s['n_ekf_features']}",
        color=STYLE['text'], fontsize=10, y=0.98)

    gs = gridspec.GridSpec(3, 4, figure=fig,
                           hspace=0.55, wspace=0.38,
                           left=0.07, right=0.97, top=0.92, bottom=0.07)

    # 1. XY pose error
    ax1 = fig.add_subplot(gs[0, :2])
    ax1.fill_between(t, 0, rec.pose_err_xy, color=STYLE['accent1'], alpha=0.3, linewidth=0)
    ax1.plot(t, rec.pose_err_xy, color=STYLE['accent1'], linewidth=0.8, label='XY error')
    if len(t) > 50:
        win = max(20, len(t)//40)
        sm  = np.convolve(rec.pose_err_xy, np.ones(win)/win, mode='valid')
        ax1.plot(t[win//2:win//2+len(sm)], sm, color=STYLE['accent3'],
                 linewidth=1.4, linestyle='--', label='Smoothed')
    ax1.axhline(s['rmse_xy'], color=STYLE['accent4'], linestyle=':',
                linewidth=1.1, label=f"RMSE={s['rmse_xy']:.3f}m")
    ax1.legend(fontsize=6, labelcolor=STYLE['text'],
               facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    _style_ax(ax1, 'Pose Error — XY (m)', 'Time (s)', 'Error (m)')

    # 2. Heading error
    ax2 = fig.add_subplot(gs[0, 2:])
    ax2.fill_between(t, 0, np.degrees(rec.pose_err_th),
                     color=STYLE['accent2'], alpha=0.3, linewidth=0)
    ax2.plot(t, np.degrees(rec.pose_err_th), color=STYLE['accent2'], linewidth=0.8)
    ax2.axhline(s['rmse_th_deg'], color=STYLE['accent4'], linestyle=':',
                linewidth=1.1, label=f"RMSE={s['rmse_th_deg']:.2f}°")
    ax2.legend(fontsize=6, labelcolor=STYLE['text'],
               facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    _style_ax(ax2, 'Pose Error — Heading (°)', 'Time (s)', 'Error (°)')

    # 3. NEES
    ax3 = fig.add_subplot(gs[1, :2])
    ax3.plot(t, rec.nees, color=STYLE['accent5'], linewidth=0.6, alpha=0.7)
    ax3.axhline(3.0, color=STYLE['accent3'], linestyle='--', linewidth=1.1,
                label='Expected (dof=3)')
    ax3.axhline(s['nees_hi'], color=STYLE['accent4'], linestyle=':',
                linewidth=0.9, label=f"95% upper={s['nees_hi']:.1f}")
    ax3.axhline(s['nees_lo'], color=STYLE['accent2'], linestyle=':',
                linewidth=0.9, label=f"95% lower={s['nees_lo']:.2f}")
    ax3.set_ylim(-0.5, min(float(np.nanpercentile(rec.nees, 98))*1.5, 50))
    ax3.legend(fontsize=6, labelcolor=STYLE['text'],
               facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    _style_ax(ax3,
              f"NEES  mean={s['nees_mean']:.2f}  "
              f"{s['frac_consistent']*100:.0f}% within 95% bounds",
              'Time (s)', 'NEES')

    # 4. Covariance trace
    ax4 = fig.add_subplot(gs[1, 2:])
    ax4.semilogy(t, rec.pose_cov_trace, color=STYLE['accent6'], linewidth=0.9)
    _style_ax(ax4, 'Vehicle Pose Covariance Trace  tr(Pvv)',
              'Time (s)', 'tr(Pvv)  [log scale]')

    # 5. Innovation chi-squared histogram
    ax5 = fig.add_subplot(gs[2, 0])
    iv  = s['innov_valid']
    if len(iv) > 0:
        hmax = min(float(np.percentile(iv, 98)), 20.0)
        ax5.hist(iv, bins=np.linspace(0, hmax, 40), density=True,
                 color=STYLE['accent1'], alpha=0.75, edgecolor='none')
        xs = np.linspace(0, hmax, 200)
        ax5.plot(xs, stats.chi2.pdf(xs, df=2), color=STYLE['accent3'],
                 linewidth=1.4, label='χ²(2) expected')
        ax5.axvline(s['innov_lo'], color=STYLE['accent2'], linestyle=':', linewidth=0.9)
        ax5.axvline(s['innov_hi'], color=STYLE['accent4'], linestyle=':',
                    linewidth=0.9, label='95% bounds')
        ax5.legend(fontsize=6, labelcolor=STYLE['text'],
                   facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    _style_ax(ax5, f"Innovation χ²  mean={s['innov_mean']:.2f}\n"
              f"{s['frac_innov_ok']*100:.0f}% within χ²(2) 95%",
              'νᵀS⁻¹ν', 'Density')

    # 6. Feature count
    ax6 = fig.add_subplot(gs[2, 1])
    feat_t = t[:len(rec.n_features)]
    ax6.step(feat_t, rec.n_features, color=STYLE['accent2'], linewidth=1.2, where='post')
    ax6.fill_between(feat_t, 0, rec.n_features, color=STYLE['accent2'],
                     alpha=0.15, step='post')
    _style_ax(ax6, f"Map Size  final={s['n_ekf_features']}", 'Time (s)', '# features')

    # 7. Feature map RMSE bars
    ax7 = fig.add_subplot(gs[2, 2])
    cats, vals, cols = [], [], []
    if not np.isnan(s['corner_rmse_cm']):
        cats.append('Corner\nXY (cm)'); vals.append(s['corner_rmse_cm'])
        cols.append(STYLE['accent1'])
    if not np.isnan(s['line_rho_rmse_cm']):
        cats.append('Line ρ\n(cm)'); vals.append(s['line_rho_rmse_cm'])
        cols.append(STYLE['accent2'])
    if not np.isnan(s['line_alpha_rmse_deg']):
        cats.append('Line α\n(deg)'); vals.append(s['line_alpha_rmse_deg'])
        cols.append(STYLE['accent3'])
    if cats:
        bars = ax7.bar(cats, vals, color=cols, edgecolor='none', alpha=0.85)
        for bar, val in zip(bars, vals):
            ax7.text(bar.get_x()+bar.get_width()/2,
                     bar.get_height()+0.01*max(vals),
                     f'{val:.2f}', ha='center', va='bottom',
                     color=STYLE['text'], fontsize=7)
    _style_ax(ax7, 'Feature Map RMSE', '', 'RMSE')

    # 8. Recall bars
    ax8 = fig.add_subplot(gs[2, 3])
    recalls = [s['recall_corners']*100, s['recall_lines']*100]
    brs = ax8.bar(['Corner\nrecall', 'Line\nrecall'], recalls,
                  color=[STYLE['accent1'], STYLE['accent2']],
                  edgecolor='none', alpha=0.85)
    for b, v in zip(brs, recalls):
        ax8.text(b.get_x()+b.get_width()/2, v+1.5, f'{v:.0f}%',
                 ha='center', va='bottom', color=STYLE['text'], fontsize=8)
    ax8.set_ylim(0, 115)
    ax8.axhline(100, color=STYLE['grid'], linestyle='--', linewidth=0.8)
    _style_ax(ax8, 'Feature Recall', '', 'Recall (%)')

    return fig


# ── Lab analytics ─────────────────────────────────────────────────────────────
cfg = Config.load('config.yaml')
summary_lab = compute_summary(rec_lab, gt_lab, state_lab, cfg)
fig_lab = plot_analytics(rec_lab, summary_lab, 'lab')
fig_lab.savefig('nb_analytics_lab.png', dpi=120, bbox_inches='tight',
                facecolor=STYLE['bg'])
display(fig_lab)
plt.close(fig_lab)

print(f"\nLab RMSE XY:       {summary_lab['rmse_xy']*100:.2f} cm")
print(f"Lab NEES mean:     {summary_lab['nees_mean']:.2f}  (expect ≈3)")
print(f"Lab corner recall: {summary_lab['recall_corners']*100:.0f}%")
print(f"Lab wall crossings:{summary_lab['wall_crossings']}")

## 9. Run the Corridor World and Compare

The **corridor world** (20×3 m with a perpendicular branch) is geometrically degenerate for the EKF. Parallel walls provide range constraints but weak **bearing** constraints, so heading error accumulates unchecked. The EKF's single-Gaussian assumption cannot represent the multi-modal heading posterior that arises in a symmetric corridor.

> ⚠️ The corridor run is capped at `MAX_STEPS=8000`. On slower machines this may take 3–5 minutes. The key results are the NEES values, which reveal whether the filter's stated confidence matches reality.

In [ ]:
cfg = Config.load('config.yaml')
t0  = time.perf_counter()

rec_corr, gt_corr, state_corr, cfg = run_simulation(
    cfg, preset='corridor', n_samples=300, seed=0, verbose=True)

print(f"\nWall-clock time: {time.perf_counter()-t0:.1f}s")
print(f"EKF features:    {state_corr.n_features}")
print(f"Path length:     {rec_corr.path_length:.2f} m")
print(f"Wall crossings:  {rec_corr.wall_crossings}")

In [ ]:
summary_corr = compute_summary(rec_corr, gt_corr, state_corr, cfg)
fig_corr = plot_analytics(rec_corr, summary_corr, 'corridor')
fig_corr.savefig('nb_analytics_corridor.png', dpi=120, bbox_inches='tight',
                 facecolor=STYLE['bg'])
display(fig_corr)
plt.close(fig_corr)

print(f"\nCorridor RMSE XY:       {summary_corr['rmse_xy']*100:.2f} cm")
print(f"Corridor NEES mean:     {summary_corr['nees_mean']:.2f}  (expect ≈3)")
print(f"Corridor corner recall: {summary_corr['recall_corners']*100:.0f}%")
print(f"Corridor wall crossings:{summary_corr['wall_crossings']}")

## 10. Head-to-Head Comparison

The most important comparison is between **NEES** and **innovation chi-squared** across the two worlds. The corridor's innovation statistics look healthy (mean ≈ 1.1, >94% within bounds) while NEES is catastrophically wrong (mean ≈ 135 vs expected 3). This is the EKF's central failure mode: **the filter certifies itself as healthy while being substantially wrong**, because innovations are evaluated at the filter's own (incorrect) state estimate.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8), facecolor=STYLE['bg'])
fig.suptitle('Lab vs Corridor — Head-to-Head Comparison',
             color=STYLE['text'], fontsize=11)

worlds   = ['lab',          'corridor']
recs     = [rec_lab,        rec_corr]
summaries= [summary_lab,    summary_corr]
colours  = [STYLE['accent2'], STYLE['accent4']]

# Row 0: XY pose error
for i, (w, rec, col) in enumerate(zip(worlds, recs, colours)):
    ax = axes[0, i] if i < 2 else axes[0, 2]
    ax = axes[0, i]
    ax.fill_between(rec.t, 0, rec.pose_err_xy, color=col, alpha=0.3, linewidth=0)
    ax.plot(rec.t, rec.pose_err_xy, color=col, linewidth=0.8)
    ax.axhline(summaries[i]['rmse_xy'], color='white', linestyle=':',
               linewidth=1.0, label=f"RMSE={summaries[i]['rmse_xy']*100:.1f}cm")
    ax.legend(fontsize=7, labelcolor=STYLE['text'],
              facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    _style_ax(ax, f"XY Pose Error — {w}", 'Time (s)', 'Error (m)')

# Row 0, col 2: RMSE bar comparison
ax = axes[0, 2]
bar_labels = ['Lab', 'Corridor']
bar_vals   = [summary_lab['rmse_xy']*100, summary_corr['rmse_xy']*100]
bars = ax.bar(bar_labels, bar_vals, color=colours, edgecolor='none', alpha=0.85)
for b, v in zip(bars, bar_vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.3, f'{v:.1f}cm',
            ha='center', va='bottom', color=STYLE['text'], fontsize=9)
_style_ax(ax, 'XY RMSE Comparison', '', 'RMSE (cm)')

# Row 1: NEES
for i, (w, rec, s, col) in enumerate(zip(worlds, recs, summaries, colours)):
    ax = axes[1, i]
    ax.plot(rec.t, rec.nees, color=col, linewidth=0.6, alpha=0.8)
    ax.axhline(3.0, color=STYLE['accent3'], linestyle='--', linewidth=1.1,
               label='Expected=3')
    ax.axhline(s['nees_hi'], color='white', linestyle=':',
               linewidth=0.9, label=f"95% upper={s['nees_hi']:.1f}")
    ax.set_ylim(-0.5, min(float(np.nanpercentile(rec.nees, 98))*1.5, 50))
    ax.legend(fontsize=6, labelcolor=STYLE['text'],
              facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    _style_ax(ax, f"NEES — {w}  (mean={s['nees_mean']:.1f})",
              'Time (s)', 'NEES')

# Row 1, col 2: NEES vs innovation summary bar chart
ax = axes[1, 2]
metrics = ['NEES\nmean', 'Innovation\nχ² mean']
lab_vals  = [summary_lab['nees_mean'],  summary_lab['innov_mean']]
corr_vals = [summary_corr['nees_mean'], summary_corr['innov_mean']]
x = np.arange(len(metrics))
bw = 0.35
b1 = ax.bar(x - bw/2, lab_vals,  bw, label='Lab',      color=colours[0], alpha=0.85)
b2 = ax.bar(x + bw/2, corr_vals, bw, label='Corridor', color=colours[1], alpha=0.85)
ax.axhline(3.0, color=STYLE['accent3'], linestyle='--', linewidth=1.0,
           label='NEES expected=3')
ax.axhline(2.0, color=STYLE['accent6'], linestyle='--', linewidth=1.0,
           label='Innov expected=2')
ax.set_xticks(x); ax.set_xticklabels(metrics, color=STYLE['text'], fontsize=8)
ax.legend(fontsize=6, labelcolor=STYLE['text'],
          facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
_style_ax(ax, 'NEES vs Innovation χ²\n(The Paradox)', '', 'Value')
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('nb_comparison.png', dpi=120, bbox_inches='tight',
            facecolor=STYLE['bg'])
display(fig)
plt.close(fig)
print('Comparison figure saved to nb_comparison.png')

## 11. Parameter Sensitivity Analysis

A key question is: how sensitive is the filter to the **process noise** parameters $Q_v$ and $Q_\omega$? These control how much uncertainty the filter injects during the prediction step. Too small → overconfident predictions; too large → innovations have less corrective power.

We run the lab world with three different $Q_\omega$ values and compare the resulting NEES and RMSE. This is a lightweight sensitivity study that takes a few minutes.

In [ ]:
Q_w_values = [0.01, 0.05, 0.20]   # low / default / high rotational noise
sensitivity_results = []

for q_w in Q_w_values:
    cfg_s = Config.load('config.yaml')
    cfg_s.ekf.Q_w = q_w
    print(f"\n── Q_w = {q_w} ──────────────────────────")
    rec_s, gt_s, state_s, _ = run_simulation(
        cfg_s, preset='lab', n_samples=200, seed=0, verbose=True)
    summ_s = compute_summary(rec_s, gt_s, state_s, cfg_s)
    sensitivity_results.append({
        'Q_w': q_w,
        'rmse_xy_cm': summ_s['rmse_xy'] * 100,
        'nees_mean':  summ_s['nees_mean'],
        'frac_consistent': summ_s['frac_consistent'],
        'n_features': summ_s['n_ekf_features'],
    })
    print(f"  RMSE XY: {summ_s['rmse_xy']*100:.2f}cm  "
          f"NEES: {summ_s['nees_mean']:.2f}  "
          f"consistent: {summ_s['frac_consistent']*100:.0f}%")

In [ ]:
q_ws        = [r['Q_w']          for r in sensitivity_results]
rmse_vals   = [r['rmse_xy_cm']   for r in sensitivity_results]
nees_vals   = [r['nees_mean']    for r in sensitivity_results]
frac_vals   = [r['frac_consistent']*100 for r in sensitivity_results]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), facecolor=STYLE['bg'])
fig.suptitle('Parameter Sensitivity — Q_ω (rotational process noise) · Lab World',
             color=STYLE['text'], fontsize=10)

for ax, vals, label, colour in zip(
        axes,
        [rmse_vals, nees_vals, frac_vals],
        ['XY RMSE (cm)', 'NEES mean', '% steps consistent'],
        [STYLE['accent1'], STYLE['accent5'], STYLE['accent2']]):
    ax.plot(q_ws, vals, 'o-', color=colour, linewidth=2.0,
            markersize=8, markerfacecolor=colour)
    for qw, v in zip(q_ws, vals):
        ax.annotate(f'{v:.2f}', (qw, v), textcoords='offset points',
                    xytext=(0, 10), ha='center',
                    color=STYLE['text'], fontsize=8)
    if label == 'NEES mean':
        ax.axhline(3.0, color=STYLE['accent3'], linestyle='--',
                   linewidth=1.0, label='Expected=3')
        ax.legend(fontsize=7, labelcolor=STYLE['text'],
                  facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    if label == '% steps consistent':
        ax.axhline(95, color=STYLE['accent3'], linestyle='--',
                   linewidth=1.0, label='Target 95%')
        ax.set_ylim(0, 105)
        ax.legend(fontsize=7, labelcolor=STYLE['text'],
                  facecolor=STYLE['panel'], edgecolor=STYLE['grid'])
    _style_ax(ax, label, 'Q_ω', label)

plt.tight_layout()
plt.savefig('nb_sensitivity.png', dpi=120, bbox_inches='tight',
            facecolor=STYLE['bg'])
display(fig)
plt.close(fig)
print('Sensitivity figure saved to nb_sensitivity.png')

## 12. Final Summary Table

All key metrics from both worlds side by side.

In [ ]:
def print_summary_table(summaries, labels):
    rows = [
        ('Simulation duration (s)',      'total_time',           '{:.1f}'),
        ('Path length (m)',              'path_length',          '{:.2f}'),
        ('Wall crossings',               'wall_crossings',       '{}'),
        ('─' * 35,                       None,                   ''),
        ('POSE ESTIMATION',              None,                   ''),
        ('  XY RMSE (cm)',               'rmse_xy',              '{:.2f}',  100),
        ('  Heading RMSE (deg)',          'rmse_th_deg',          '{:.3f}'),
        ('  Max XY error (cm)',          'max_xy',               '{:.2f}',  100),
        ('  Final XY error (cm)',        'final_xy',             '{:.2f}',  100),
        ('─' * 35,                       None,                   ''),
        ('FILTER CONSISTENCY',           None,                   ''),
        ('  NEES mean (expect ≈3)',      'nees_mean',            '{:.2f}'),
        ('  % steps in χ²(3) 95%',      'frac_consistent',      '{:.0f}%', 100),
        ('  Innovation χ² (expect ≈2)', 'innov_mean',           '{:.2f}'),
        ('  % innov in χ²(2) 95%',      'frac_innov_ok',        '{:.0f}%', 100),
        ('─' * 35,                       None,                   ''),
        ('MAP QUALITY',                  None,                   ''),
        ('  EKF features (total)',       'n_ekf_features',       '{}'),
        ('  Corner recall',              'recall_corners',       '{:.0f}%', 100),
        ('  Line recall',                'recall_lines',         '{:.0f}%', 100),
        ('  Corner RMSE (cm)',           'corner_rmse_cm',       '{:.2f}'),
        ('  Line ρ RMSE (cm)',           'line_rho_rmse_cm',     '{:.2f}'),
        ('  Line α RMSE (deg)',          'line_alpha_rmse_deg',  '{:.3f}'),
        ('  Data assoc match rate',      'match_rate',           '{:.1f}%', 100),
    ]

    col_w = 36
    val_w = 14
    header = f"{'Metric':<{col_w}}" + "".join(f"{l:>{val_w}}" for l in labels)
    print(header)
    print('─' * (col_w + val_w * len(labels)))

    for row in rows:
        label = row[0]
        key   = row[1]
        fmt   = row[2]
        scale = row[3] if len(row) > 3 else 1

        if key is None:
            if label.startswith('─'):
                print('─' * (col_w + val_w * len(labels)))
            else:
                print(f"\n{label}")
            continue

        vals_str = []
        for s in summaries:
            v = s.get(key, np.nan)
            if isinstance(v, float) and np.isnan(v):
                vals_str.append('N/A')
            else:
                vals_str.append(fmt.format(v * scale if scale != 1 else v))
        print(f"{label:<{col_w}}" + "".join(f"{v:>{val_w}}" for v in vals_str))


print_summary_table(
    [summary_lab, summary_corr],
    ['Lab', 'Corridor']
)

## 13. Key Takeaways

### The EKF-SLAM paradox in the corridor

The most important result in this notebook is the gap between **NEES** and **innovation chi-squared** in the corridor world. Both metrics should be approximately equal to their degrees of freedom (3 and 2 respectively) for a well-calibrated filter. In the lab world, both are close to expected. In the corridor:

- **Innovation χ²** ≈ 1.1 — looks healthy, 94% within bounds
- **NEES** ≈ 135 — catastrophically wrong, only 5% within bounds

This is the EKF's most dangerous failure mode. Innovations are computed at the filter's own (incorrect) estimate, so as long as the robot re-observes the same wrong landmarks from the same biased position, the innovation appears small. **Internal diagnostics cannot detect external inconsistency.** NEES requires ground-truth comparison and is the only reliable validation metric.

### Why this matters for algorithm choice

| Setting | EKF | Particle Filter | Graph SLAM |
|---------|-----|-----------------|------------|
| Open room, diverse features | ✓ Excellent | ✓ Good (more expensive) | ✓ Good |
| Long symmetric corridor | ✗ Overconfident | ✓ Can maintain multi-modal heading | ✓ Retroactive correction |
| Real-time online mapping | ✓ O(n²) per step | ~ O(N·n²) particles | ~ O(n²) incremental |
| Can detect own failure | ✗ No | ✓ Particle diversity signals collapse | ✓ Loop-closure residuals |

The EKF is the right choice when geometry is well-conditioned and computational budget is limited. For corridor-like or ambiguous environments, a particle filter or graph-based backend is required.